In [35]:
from datetime import datetime, timedelta
import urllib
import requests
import xml.etree.ElementTree as ET
import time
import pandas as pd

https://arxiv.org/search/advanced?advanced=&terms-0-operator=AND&terms-0-term=protein*&terms-0-field=title&terms-1-operator=OR&terms-1-term=peptide*&terms-1-field=title&terms-2-operator=OR&terms-2-term=nanobod*&terms-2-field=title&terms-3-operator=OR&terms-3-term=antibod*&terms-3-field=title&terms-4-operator=OR&terms-4-term=enzyme*&terms-4-field=title&terms-5-operator=AND&terms-5-term=T-cell+receptor&terms-5-field=title&terms-6-operator=AND&terms-6-term=TCR&terms-6-field=title&classification-physics_archives=all&classification-include_cross_list=include&date-year=&date-from_date=2025-04-25&date-to_date=2025-05-04&date-date_type=submitted_date_first&abstracts=show&size=200&order=-announced_date_first

In [39]:

def next_day(date):

    date_obj = datetime.strptime(date, "%Y/%m/%d")
    next_day = date_obj + timedelta(days=1)
    next_day = next_day.strftime("%Y/%m/%d")
    
    return next_day


def construct_arxiv_api_query(date):
    """
    Constructs an arXiv API query URL from the search parameters.
    
    Args:
        search_params (dict): Dictionary containing search parameters
    
    Returns:
        str: The constructed API query URL
    """
    base_url = "http://export.arxiv.org/api/query?"
    
    # Construct the search query
    query_parts = []
    
    # Add title search terms
    title_terms = []
    for term in ["protein*", "peptide*", "nanobod*", "antibod*", "enzyme*","T-cell receptor*", "TCR*"]:
        title_terms.append(f"ti:{term}")
    
    # Combine title terms with OR
    if title_terms:
        query_parts.append("(" + " OR ".join(title_terms) + ")")
    search_query=''.join(query_parts)
    
    # Add date range
    date_from = date
    date_to = next_day(date)
    
    # Format dates for arXiv API (YYYYMMDD format)
    date_from_formatted = date_from.replace('/', '')
    date_to_formatted = date_to.replace('/', '')
    
    search_query += f" AND submittedDate:[{date_from_formatted} TO {date_to_formatted}]"
    
    # URL encode the query
    encoded_query = urllib.parse.quote(search_query)
    
    # Construct the final URL with parameters
    params = {
        "search_query": encoded_query,
        "start": 0,
        "max_results": 200,
        "sortBy": "submittedDate",
        "sortOrder": "descending"
    }
    
    query_string = "&".join([f"{k}={v}" for k, v in params.items()])
    
    final_query=base_url + query_string
    return final_query

def fetch_arxiv_results(api_url, max_retries=3, retry_delay=5):
    """
    Fetches results from the arXiv API with retry mechanism.
    
    Args:
        api_url (str): The API URL to fetch results from
        max_retries (int): Maximum number of retry attempts
        retry_delay (int): Delay between retries in seconds
    
    Returns:
        list: List of dictionaries containing article information
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    for attempt in range(max_retries):
        try:
            response = requests.get(api_url, headers=headers, timeout=30)
            response.raise_for_status()
            
            # Parse the XML response
            root = ET.fromstring(response.content)
            
            # Define the XML namespace
            ns = {'atom': 'http://www.w3.org/2005/Atom',
                  'arxiv': 'http://arxiv.org/schemas/atom'}
            
            # Extract entries
            entries = root.findall('.//atom:entry', ns)
            
            results = []
            for entry in entries:
                # Extract basic metadata
                title = entry.find('./atom:title', ns).text.strip()
                abstract = entry.find('./atom:summary', ns).text.strip()
                published = entry.find('./atom:published', ns).text.split('T')[0]
                
                # Extract DOI if available
                doi = None
                for link in entry.findall('./atom:link', ns):
                    if link.get('title') == 'doi':
                        doi = link.get('href')
                
                # Extract arXiv ID
                id_url = entry.find('./atom:id', ns).text
                arxiv_id = id_url.split('/')[-1]
                
                # If no DOI is found, create the arXiv DOI
                if not doi:
                    doi = f"https://doi.org/10.48550/arXiv.{arxiv_id.split('v')[0]}"

                
                results.append({
                    'title': title,
                    'abstract': abstract,
                    'date': published,
                    'doi': doi,
                })
            
            return results
        
        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                print(f"Error fetching results (attempt {attempt+1}/{max_retries}): {e}")
                print(f"Retrying in {retry_delay} seconds...")
                time.sleep(retry_delay)
            else:
                print(f"Failed to fetch results after {max_retries} attempts: {e}")
                return []
                
def list_dates(start_date, end_date):
    """
    List all dates between the start and end date, inclusive.
    
    Parameters:
    - start_date: Start date in the format 'YYYY/MM/DD'
    - end_date: End date in the format 'YYYY/MM/DD'
    
    Returns:
    - List of dates as strings in 'YYYY/MM/DD' format
    """
    # Convert string dates to datetime objects
    start = datetime.strptime(start_date, '%Y/%m/%d')
    end = datetime.strptime(end_date, '%Y/%m/%d')
    
    # Create a list of dates
    date_list = []
    current_date = start
    
    while current_date <= end:
        date_list.append(current_date.strftime('%Y/%m/%d'))
        current_date += timedelta(days=1)  # Increment by one day
    
    return date_list


def run_arxiv_search(start_date, end_date):
    
    all_results=[]
    
    dates_list=list_dates(start_date, end_date)
    
    for date in dates_list:
        # Construct the API query URL
        api_url = construct_arxiv_api_query(date)
        
        # Fetch the results
        print(f"Fetching results from arXiv API for date {date}...")
        results = fetch_arxiv_results(api_url)
        
        # Print the results
        print(f"\nFound {len(results)} articles for date {date}\n")
        
        for article in results:
            article['search_date']=date.replace('/','-')
            all_results.append(article)
            
    return pd.DataFrame(all_results)

    


In [41]:
start_date, end_date = "2025/04/25","2025/05/04"
df=run_arxiv_search(start_date, end_date)
df

Fetching results from arXiv API for date 2025/04/25...

Found 2 articles for date 2025/04/25

Fetching results from arXiv API for date 2025/04/26...

Found 1 articles for date 2025/04/26

Fetching results from arXiv API for date 2025/04/27...

Found 2 articles for date 2025/04/27

Fetching results from arXiv API for date 2025/04/28...

Found 0 articles for date 2025/04/28

Fetching results from arXiv API for date 2025/04/29...

Found 2 articles for date 2025/04/29

Fetching results from arXiv API for date 2025/04/30...

Found 0 articles for date 2025/04/30

Fetching results from arXiv API for date 2025/05/01...

Found 0 articles for date 2025/05/01

Fetching results from arXiv API for date 2025/05/02...

Found 0 articles for date 2025/05/02

Fetching results from arXiv API for date 2025/05/03...

Found 0 articles for date 2025/05/03

Fetching results from arXiv API for date 2025/05/04...

Found 0 articles for date 2025/05/04



,title,abstract,date,doi,search_date
0,CAML: Commutative algebra machine learning -- ...,"Recently, Suwayyid and Wei have introduced com...",2025-04-25,https://doi.org/10.48550/arXiv.2504.18646,2025/04/25
1,"Enhanced Sampling, Public Dataset and Generati...",Drug-protein binding and dissociation dynamics...,2025-04-25,https://doi.org/10.48550/arXiv.2504.18367,2025/04/25
2,Sparks: Multi-Agent Artificial Intelligence Mo...,Advances in artificial intelligence (AI) promi...,2025-04-26,https://doi.org/10.48550/arXiv.2504.19017,2025/04/26
3,Metric Similarity and Manifold Learning of Cir...,We present a machine learning analysis of circ...,2025-04-27,https://doi.org/10.48550/arXiv.2504.19355,2025/04/27
4,HyboWaveNet: Hyperbolic Graph Neural Networks ...,Protein-protein interactions (PPIs) are fundam...,2025-04-27,https://doi.org/10.48550/arXiv.2504.20102,2025/04/27
5,ProT-GFDM: A Generative Fractional Diffusion M...,This work introduces the generative fractional...,2025-04-29,https://doi.org/10.48550/arXiv.2504.21092,2025/04/29
6,BEER: Biochemical Estimator and Explorer of Re...,Protein sequence analysis underpins research i...,2025-04-29,https://doi.org/10.48550/arXiv.2504.20561,2025/04/29
